# REMNET · RealWaste D3 · Section 2 — Dataset Independence Audit (D1 + D3 ONLY)

Adapted from the user's Stage-3 Section-2 reference notebook (`s3p2(1).ipynb`): complete image counts and integrity, SHA-256 duplicate screening, exhaustive pHash alerts, automated visual-comparison evidence, conservative removal/withholding, six-class mapping, and a fingerprinted final D3 image manifest.

**Only two Kaggle inputs are used:**

- D3: `/kaggle/input/datasets/akash264208/real-waste`
- Original D1: `/kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset`

**No Stage 2 split manifest is required or loaded.** All accessible original D1 source images in the six categories are scanned directly, including any available training/validation/test folders. When original split assignments are not encoded in image paths, their exact memberships are reported as **UNKNOWN**. A full-source duplicate audit does not independently establish that the historical 1,956/487/463 partitions have been reconstructed.

**Conservative policy:** exclude exact D1 overlaps, unreadable D3 images and redundant D3 byte copies. Withhold all images appearing in any 64-bit pHash candidate pair (Hamming distance ≤6), regardless of machine visual similarity. Contact sheets and automated image-level examination are retained. Automated processing is **not** claimed as human review of image pairs. Only the RealWaste-to-REMNET class mapping was confirmed by the user.

The final evaluation manifest, if lock checks pass, will be **generated by this notebook**, not read from a supplied file. Use the identical resulting D3 manifest for all four model configurations and five seeds. No training, fine-tuning, classification inference, or CodeCarbon is performed here.


In [2]:
# 1 — D3 RealWaste + original D1 only (NO external manifest file)
from pathlib import Path
from collections import Counter, defaultdict
import os, csv, json, hashlib, zipfile, math, datetime, platform, shutil, re
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from scipy.fft import dctn

D3_INPUT_PATH = Path('/kaggle/input/datasets/akash264208/real-waste')
D1_INPUT_PATH = Path('/kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset')
# Optional synthetic-test overrides only; unset these when running in Kaggle.
D3_INPUT_PATH = Path(os.environ.get('REMNET_D3_ROOT_OVERRIDE', str(D3_INPUT_PATH)))
D1_INPUT_PATH = Path(os.environ.get('REMNET_D1_ROOT_OVERRIDE', str(D1_INPUT_PATH)))
OUT = Path('/kaggle/working/REMNET_D3_Section2_Independence') if Path('/kaggle').exists() else Path('/mnt/data/REMNET_D3_Section2_Independence')
OUT = Path(os.environ.get('REMNET_D3_SECTION2_OUT', str(OUT)))
OUT.mkdir(parents=True, exist_ok=True)

MODEL_CLASSES=('cardboard','glass','metal','paper','plastic','trash')
CLASS_TO_INDEX={label:i for i,label in enumerate(MODEL_CLASSES)}
D3_SOURCE_TO_MODEL={
    'Cardboard':'cardboard', 'Glass':'glass', 'Metal':'metal',
    'Paper':'paper', 'Plastic':'plastic', 'Miscellaneous Trash':'trash'
}
D3_EXCLUDED_SOURCE=('Food Organics','Textile Trash','Vegetation')
ALL_D3_CLASSES=tuple(D3_SOURCE_TO_MODEL)+D3_EXCLUDED_SOURCE
PUBLISHED_D3_COUNTS={
 'Cardboard':461,'Food Organics':411,'Glass':420,'Metal':790,
 'Miscellaneous Trash':495,'Paper':500,'Plastic':921,'Textile Trash':318,'Vegetation':436
}
IMAGE_EXTENSIONS={'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
PHASH_HAMMING_THRESHOLD=6
PAIRS_PER_SHEET=8
MAX_CONTACT_SHEETS=500
D3_CLASS_MAPPING_HUMAN_REVIEW='CONFIRMED_BY_USER'
EXPECTED_D1_SPLITS={'Training':1956,'Validation':487,'Testing':463} # descriptive only; NEVER fabricate assignments

print('D3 RealWaste:', D3_INPUT_PATH)
print('D1 original source:', D1_INPUT_PATH)
print('Stage 2 manifest input: NOT USED / NOT REQUIRED')
print('Output:', OUT)


D3 RealWaste: /kaggle/input/datasets/akash264208/real-waste
D1 original source: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset
Stage 2 manifest input: NOT USED / NOT REQUIRED
Output: /kaggle/working/REMNET_D3_Section2_Independence


## 2. Locate RealWaste categories and original D1 images

Use the supplied D3 and D1 mounted folders only. Search nested class directories and safely extract embedded archives if necessary. **Every original D1 source tree** containing the six class directories is screened; an external split-manifest file is not consulted.


In [3]:
# 2 — Safe archive fallback, class-root discovery and source location

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as fp:
        for block in iter(lambda:fp.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def image_files(root):
    return sorted((p for p in Path(root).rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS), key=lambda p:p.as_posix())

def safe_extract(src, dest):
    dest.mkdir(parents=True,exist_ok=True)
    target_base=dest.resolve()
    with zipfile.ZipFile(src) as z:
        for info in z.infolist():
            final=(target_base / info.filename).resolve()
            if not final.is_relative_to(target_base):
                raise RuntimeError('Unsafe ZIP archive member: '+info.filename)
        z.extractall(target_base)
    return dest

def category_root_candidates(root, category_names):
    expected={x.lower() for x in category_names}
    matches=[]
    for parent, dirs, _ in os.walk(root):
        keyed={x.lower():x for x in dirs}
        if expected.issubset(keyed):
            mapping={name:Path(parent)/keyed[name.lower()] for name in category_names}
            matches.append((Path(parent),mapping))
            dirs[:] = []
    return matches

def detect_roots(root, names, label):
    if not root.exists():
        return []
    if root.is_file() and root.suffix.lower()=='.zip':
        archive_dir=OUT/'extracted_inputs'/label/'main'
        if not archive_dir.is_dir() or not any(archive_dir.iterdir()):
            safe_extract(root,archive_dir)
        root=archive_dir
    hits=category_root_candidates(root,names)
    if not hits and root.is_dir():
        for i,p in enumerate(sorted(root.rglob('*.zip'))):
            dest=OUT/'extracted_inputs'/label/f'nested_{i:02d}'
            if not dest.is_dir() or not any(dest.iterdir()):
                safe_extract(p,dest)
            hits.extend(category_root_candidates(dest,names))
    return hits

D3_ROOTS=detect_roots(D3_INPUT_PATH,ALL_D3_CLASSES,'d3_realwaste')
if len(D3_ROOTS)!=1:
    raise RuntimeError(f'Exactly one RealWaste nine-class root required. Found {len(D3_ROOTS)}; path={D3_INPUT_PATH}. '
                       'Attach the D3 dataset or correct D3_INPUT_PATH. Candidate roots='+str([str(t[0]) for t in D3_ROOTS]))
D3_CLASS_ROOT, D3_CLASS_FOLDERS=D3_ROOTS[0]
print('Resolved RealWaste category root:', D3_CLASS_ROOT)

# D1 is a reference only. Scan every six-class folder tree (train/valid/test if present).
D1_ROOTS=detect_roots(D1_INPUT_PATH,MODEL_CLASSES,'original_d1')
D1_ROOT_SELECTED=D1_INPUT_PATH
D1_SOURCE_FOLDERS={c:[] for c in MODEL_CLASSES}
for root,folders in D1_ROOTS:
    print('D1 source tree discovered:',root)
    for c in MODEL_CLASSES:
        D1_SOURCE_FOLDERS[c].append(folders[c])
if not D1_ROOTS:
    print('WARNING: No complete original D1 six-class tree found; D3 lock will be blocked.')
if any(not D1_SOURCE_FOLDERS[c] for c in MODEL_CLASSES):
    print('WARNING: Missing original D1 source category folder.')

# Input fingerprints are content-derived; Kaggle dataset revision is not assumed from the mount path.
print('Original RealWaste source (UCI): https://archive.ics.uci.edu/dataset/908/realwaste')
print('Actual Kaggle mirror revision: unknown from mounted folder path alone')


Resolved RealWaste category root: /kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste
D1 source tree discovered: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Testing
D1 source tree discovered: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Training
Original RealWaste source (UCI): https://archive.ics.uci.edu/dataset/908/realwaste
Actual Kaggle mirror revision: unknown from mounted folder path alone


## 3. Verify all D3 images, compute SHA-256 and pHash

Every selected six-class image is checked for decoding integrity. Counts are determined from the **attached dataset**, not copied from UCI's published counts. Excluded D3 categories remain in the full inventory but are not eligible for evaluation.

In [4]:
# 3 — D3 complete inventory, image integrity and pHash

def perceptual_hash(path):
    """64-bit DCT pHash, 32x32 grayscale, 8x8 low-frequency block, DC removed for median."""
    with Image.open(path) as image:
        pixels=np.asarray(ImageOps.exif_transpose(image).convert('L').resize((32,32),Image.Resampling.LANCZOS),dtype=np.float32)
    low=dctn(pixels,type=2,norm='ortho')[:8,:8]
    med=np.median(low.reshape(-1)[1:])
    bits=(low>med).reshape(-1)
    result=0
    for bit in bits:
        result=(result<<1)|int(bit)
    return result

def inspect_image(path,include_phash=True):
    info={'sha256':'','phash_hex':'','image_ok':False,'decode_error':'','width':'','height':'','format':''}
    try:
        info['sha256']=sha256_file(path)
        with Image.open(path) as im:
            info['width'],info['height']=im.size
            info['format']=im.format or ''
            im.verify()
        with Image.open(path) as im:
            im.load()
        info['image_ok']=True
        if include_phash:
            info['phash_hex']=f'{perceptual_hash(path):016x}'
    except Exception as exc:
        info['decode_error']=f'{type(exc).__name__}: {exc}'[:350]
    return info

D3_ROWS=[]
D3_COUNTS={}
for category in ALL_D3_CLASSES:
    paths=image_files(D3_CLASS_FOLDERS[category])
    D3_COUNTS[category]=len(paths)
    include=category in D3_SOURCE_TO_MODEL
    print(f'D3 {category:22s} {len(paths):5d} image files')
    for p in paths:
        label=D3_SOURCE_TO_MODEL.get(category,'')
        info=inspect_image(p,include_phash=include)
        D3_ROWS.append({
            'dataset':'D3_RealWaste',
            'source_category':category, 'class':label,
            'output_index':CLASS_TO_INDEX[label] if label else '',
            'selected_six_class':include,
            'filename':p.name,'external_relative_path':p.relative_to(D3_CLASS_ROOT).as_posix(),
            'external_absolute_path':str(p.resolve()), 'file_bytes':p.stat().st_size,
            **info
        })
D3_SELECTED=[r for r in D3_ROWS if r['selected_six_class']]
D3_VALID=[r for r in D3_SELECTED if r['image_ok'] and r['phash_hex']]
D3_BAD=[r for r in D3_SELECTED if not r['image_ok'] or not r['phash_hex']]
COUNT_MATCHES_PUBLIC=all(D3_COUNTS.get(k)==v for k,v in PUBLISHED_D3_COUNTS.items())
D3_INVENTORY_FINGERPRINT=hashlib.sha256(''.join(
    f"{r['external_relative_path']}|{r['sha256']}|{r['file_bytes']}\n"
    for r in sorted(D3_ROWS,key=lambda z:z['external_relative_path'])
).encode('utf-8')).hexdigest()
print('Full nine-class file count:',sum(D3_COUNTS.values()))
print('Selected six-class raw count:',len(D3_SELECTED),'readable:',len(D3_VALID),'unreadable:',len(D3_BAD))
print('Counts match UCI published reference:',COUNT_MATCHES_PUBLIC)
print('D3 inventory SHA256:',D3_INVENTORY_FINGERPRINT)

D3 Cardboard                461 image files
D3 Glass                    420 image files
D3 Metal                    790 image files
D3 Paper                    500 image files
D3 Plastic                  921 image files
D3 Miscellaneous Trash      495 image files
D3 Food Organics            411 image files
D3 Textile Trash            318 image files
D3 Vegetation               436 image files
Full nine-class file count: 4752
Selected six-class raw count: 3587 readable: 3587 unreadable: 0
Counts match UCI published reference: True
D3 inventory SHA256: df27e8bb5683f79e3975316b6f2f6866ac2e79fa85e5535b1cd1d3fc585766d6


## 4. Build a D1 source-image reference directly (no input manifest)

Read and hash every accessible D1 source image under the six trained categories. If path names explicitly identify training/validation/testing, save those tags; otherwise record `UNKNOWN`. The audit compares D3 with **all D1 source images** regardless of split labels, which is conservative for overlap detection. Historical partition counts cannot be independently verified from this source alone if splits are not recorded in its directory structure.


In [5]:
# 4 — Direct D1 source image inventory; no Stage 2 split-manifest file

def infer_original_split(path):
    # Infer only from an explicit source directory name, never by quotas, hash or random reassignment.
    names=[part.lower().strip().replace('-','_') for part in Path(path).parts]
    names=[re.sub(r'[^a-z0-9_]','',n) for n in names]
    split_tags={'Training':{'train','training'},
                'Validation':{'val','valid','validation'},
                'Testing':{'test','testing'}}
    found=[split for split,aliases in split_tags.items() if any(n in aliases for n in names)]
    return found[0] if len(found)==1 else 'UNKNOWN'

D1_ROWS=[]
for label in MODEL_CLASSES:
    sources=[]
    for folder in D1_SOURCE_FOLDERS[label]:
        sources.extend(image_files(folder))
    paths=sorted(set(p.resolve() for p in sources),key=str)
    print(f'D1 original source {label:10s}: {len(paths):5d} image files')
    for p in paths:
        info=inspect_image(p,include_phash=True)
        D1_ROWS.append({
            'class':label,'split_inferred_from_folder':infer_original_split(p.relative_to(D1_INPUT_PATH) if p.is_relative_to(D1_INPUT_PATH) else p),
            'reference_absolute_path':str(p),
            'reference_relative_path':p.relative_to(D1_INPUT_PATH).as_posix() if p.is_relative_to(D1_INPUT_PATH) else str(p),
            **info
        })
D1_BY_SHA=defaultdict(list)
for r in D1_ROWS:
    if r['sha256'] and r['image_ok'] and r['phash_hex']:
        D1_BY_SHA[r['sha256']].append(r)

D1_INTEGRITY_ERRORS=[r for r in D1_ROWS if not r['image_ok'] or not r['phash_hex']]
D1_SOURCE_CLASSES=set(r['class'] for r in D1_ROWS)
D1_REFERENCE_READY=(bool(D1_ROWS) and D1_SOURCE_CLASSES==set(MODEL_CLASSES) and not D1_INTEGRITY_ERRORS)
D1_SPLIT_OBSERVED=Counter(r['split_inferred_from_folder'] for r in D1_ROWS)
D1_HAS_UNKNOWN_SPLITS=bool(D1_SPLIT_OBSERVED.get('UNKNOWN'))
D1_SPLIT_PARTITION_VERIFIED=(not D1_HAS_UNKNOWN_SPLITS and
    all(D1_SPLIT_OBSERVED.get(k)==v for k,v in EXPECTED_D1_SPLITS.items()) and
    sum(D1_SPLIT_OBSERVED.values())==sum(EXPECTED_D1_SPLITS.values()))
# Not an input file: reference rows are constructed here from source images.
D1_PHASH_ROWS=sorted((r for r in D1_ROWS if r['image_ok'] and r['phash_hex']),
                     key=lambda r:r['reference_relative_path'])
D1_ROW_BY_REL={r['reference_relative_path']:r for r in D1_PHASH_ROWS}
print('D1 original source-image records:',len(D1_ROWS))
print('D1 distinct readable SHA256:',len(D1_BY_SHA))
print('D1 split folder tags:',dict(D1_SPLIT_OBSERVED))
print('D1 historical partition counts VERIFIED from folders:',D1_SPLIT_PARTITION_VERIFIED)
print('D1 complete source-level duplicate reference ready:',D1_REFERENCE_READY)
if D1_HAS_UNKNOWN_SPLITS:
    print('NOTE: D1 split membership unavailable from paths. Source-level D1 screening still proceeds without assigning partitions.')
for r in D1_INTEGRITY_ERRORS[:10]:print('D1 integrity blocker:',r['reference_relative_path'],r['decode_error'])


D1 original source cardboard :   403 image files
D1 original source glass     :   629 image files
D1 original source metal     :   452 image files
D1 original source paper     :   594 image files
D1 original source plastic   :   545 image files
D1 original source trash     :   349 image files
D1 original source-image records: 2972
D1 distinct readable SHA256: 2913
D1 split folder tags: {'Testing': 464, 'Training': 2508}
D1 historical partition counts VERIFIED from folders: False
D1 complete source-level duplicate reference ready: True


## 5. Exact SHA-256 overlap screening

Check all selected six-class D3 images against all original D1 images, and check byte-identical copies inside D3. Source-level overlaps are excluded. If D1 has explicit partition folder names, include those labels in the evidence; otherwise the D1 split field remains `UNKNOWN`.


In [6]:
# 5 — Exact cross-D1 and within-D3 duplicate evidence
D3_BY_SHA=defaultdict(list)
for r in D3_VALID:D3_BY_SHA[r['sha256']].append(r)
D3_EXACT_INTERNAL={h:sorted(rs,key=lambda x:x['external_relative_path'])
                   for h,rs in D3_BY_SHA.items() if len(rs)>1}
EXACT_D1_REFERENCE_OVERLAPS=[]
D3_REASONS=defaultdict(set)
for r in D3_BAD:
    D3_REASONS[r['external_relative_path']].add('EXCLUDE_UNREADABLE_OR_PHASH_FAILED')
for r in D3_VALID:
    rel=r['external_relative_path'];digest=r['sha256']
    for original in D1_BY_SHA.get(digest,[]):
        EXACT_D1_REFERENCE_OVERLAPS.append({
           'd3_relative_path':rel,'d3_class':r['class'],'d3_sha256':digest,
           'd1_reference_path':original['reference_relative_path'],
           'd1_class':original['class'],
           'd1_split_inferred_from_folder':original['split_inferred_from_folder'],
           'label_conflict':original['class']!=r['class']
        })
        D3_REASONS[rel].add('EXCLUDE_EXACT_ORIGINAL_D1_SOURCE')
for digest,records in D3_EXACT_INTERNAL.items():
    for duplicate in records[1:]:
        D3_REASONS[duplicate['external_relative_path']].add('EXCLUDE_EXACT_INTERNAL_D3_COPY')
print('Exact D3-vs-D1 original-source match pairs:',len(EXACT_D1_REFERENCE_OVERLAPS))
print('By source folder split (UNKNOWN allowed):',
      dict(Counter(r['d1_split_inferred_from_folder'] for r in EXACT_D1_REFERENCE_OVERLAPS)))
print('D3 exact internal-duplicate groups:',len(D3_EXACT_INTERNAL))


Exact D3-vs-D1 original-source match pairs: 0
By source folder split (UNKNOWN allowed): {}
D3 exact internal-duplicate groups: 0


## 6. Exhaustive 64-bit pHash screening — D3 vs all available D1 source images

Use the same bounded-memory threshold-based comparison as the D2 reference notebook. Compare every decodable D3 candidate with **all** available D1 reference images and screen D3 against itself. Suspected pairs are withheld conservatively, not automatically declared confirmed duplicates.


In [7]:
# 6 — Full D3↔D1 and D3↔D3 candidate screens, with an explicit scan-completeness condition
# 16-bit lookup table avoids high memory use; no pair is sampled.
POPCOUNT16=np.fromiter((x.bit_count() for x in range(65536)),dtype=np.uint8,count=65536)

def hamming_block(a,b):
    av=np.asarray(a,dtype=np.uint64).reshape(-1,1)
    bv=np.asarray(b,dtype=np.uint64).reshape(1,-1)
    xx=np.ascontiguousarray(np.bitwise_xor(av,bv))
    return POPCOUNT16[xx.view(np.uint16).reshape(xx.shape[0],xx.shape[1],4)].sum(axis=2)

def find_near_pairs(a,b,within=False,threshold=6,chunk=64):
    vals_b=np.array([int(r['phash_hex'],16) for r in b],dtype=np.uint64)
    hits=[]
    for start in range(0,len(a),chunk):
        sub=a[start:start+chunk]
        vals_a=np.array([int(r['phash_hex'],16) for r in sub],dtype=np.uint64)
        dd=hamming_block(vals_a,vals_b)
        ii,jj=np.where(dd<=threshold)
        for i,j in zip(ii,jj):
            absolute_i=start+int(i); j=int(j)
            if within and absolute_i>=j:
                continue
            hits.append((absolute_i,j,int(dd[i,j])))
    return hits

CROSS_PAIR_ROWS=[]; INTERNAL_PAIR_ROWS=[]
PHASH_CROSS_COMPLETE=False; PHASH_INTERNAL_COMPLETE=False
if D1_REFERENCE_READY and D1_PHASH_ROWS:
    candidate_indices=find_near_pairs(D3_VALID,D1_PHASH_ROWS,threshold=PHASH_HAMMING_THRESHOLD)
    for i,j,dist in candidate_indices:
        a,b=D3_VALID[i],D1_PHASH_ROWS[j]
        CROSS_PAIR_ROWS.append({
            'pair_id':f'CROSS_{len(CROSS_PAIR_ROWS)+1:06d}',
            'd3_relative_path':a['external_relative_path'],'d3_class':a['class'],'d3_sha256':a['sha256'],
            'd1_reference_path':b['reference_relative_path'],'d1_class':b['class'],
            'd1_split':b['split_inferred_from_folder'],
            'd1_sha256':b['sha256'],'phash_hamming':dist,
            'pixel_identity':'','visual_review_status':'NOT_HUMAN_REVIEWED',
            'visual_comparison_ssim':'','visual_comparison_rgb_mae':''
        })
        D3_REASONS[a['external_relative_path']].add('WITHHOLD_NEAR_D1_PHASH_CANDIDATE')
    PHASH_CROSS_COMPLETE=True
else:
    print('BLOCKER: D1 original image reference is missing or has integrity errors; cross-D1 pHash cannot be certified.')

within_indices=find_near_pairs(D3_VALID,D3_VALID,within=True,threshold=PHASH_HAMMING_THRESHOLD)
for i,j,dist in within_indices:
    a,b=D3_VALID[i],D3_VALID[j]
    INTERNAL_PAIR_ROWS.append({
        'pair_id':f'WITHIN_{len(INTERNAL_PAIR_ROWS)+1:06d}',
        'd3_image_a':a['external_relative_path'],'d3_class_a':a['class'],'d3_image_b':b['external_relative_path'],
        'd3_class_b':b['class'],'sha256_a':a['sha256'],'sha256_b':b['sha256'],
        'phash_hamming':dist,'pixel_identity':'','visual_review_status':'NOT_HUMAN_REVIEWED',
        'visual_comparison_ssim':'','visual_comparison_rgb_mae':''
    })
    # Conservative: withhold both; no guessed distinct-image classification.
    if a['sha256']!=b['sha256']:
        D3_REASONS[a['external_relative_path']].add('WITHHOLD_NEAR_INTERNAL_D3_PHASH_CANDIDATE')
        D3_REASONS[b['external_relative_path']].add('WITHHOLD_NEAR_INTERNAL_D3_PHASH_CANDIDATE')
PHASH_INTERNAL_COMPLETE=True
print('External-to-original pHash candidate pairs:',len(CROSS_PAIR_ROWS))
print('Within-D3 pHash candidate pairs:',len(INTERNAL_PAIR_ROWS))
print('Cross-D1 screen complete:',PHASH_CROSS_COMPLETE,'Within-D3 screen complete:',PHASH_INTERNAL_COMPLETE)


External-to-original pHash candidate pairs: 10
Within-D3 pHash candidate pairs: 58
Cross-D1 screen complete: True Within-D3 screen complete: True


## 7. Inspect suspected pairs: decoded pixels, visual similarity and contact sheets

The code opens **every** pHash candidate, compares decoded pixels, calculates an image-level similarity score where `scikit-image` is available, and creates side-by-side contact sheets for human inspection. **Machine-derived scores do not substitute for a documented human decision.** The default locked cohort conservatively withholds every unadjudicated pHash candidate.

In [8]:
# 7 — Machine-assisted visual examination and review contact sheets for all candidates
VISUAL_DIR=OUT/'D3_Section2_Suspected_Pair_Contact_Sheets'
VISUAL_DIR.mkdir(parents=True,exist_ok=True)
try:
    from skimage.metrics import structural_similarity as structural_similarity
    SKIMAGE_AVAILABLE=True
except ImportError:
    structural_similarity=None
    SKIMAGE_AVAILABLE=False

D3_ROW_BY_REL={r['external_relative_path']:r for r in D3_VALID}
D1_ROW_BY_REL={r['reference_relative_path']:r for r in D1_PHASH_ROWS}
PIXEL_DIGEST_CACHE={}

def decoded_pixel_hash(path):
    key=str(path)
    if key not in PIXEL_DIGEST_CACHE:
        with Image.open(path) as im:
            rgb=ImageOps.exif_transpose(im).convert('RGB')
            h=hashlib.sha256()
            h.update(f'{rgb.width}x{rgb.height};RGB'.encode('utf-8'))
            h.update(rgb.tobytes())
            PIXEL_DIGEST_CACHE[key]=h.hexdigest()
    return PIXEL_DIGEST_CACHE[key]

def normalized_pixels(path,size=(160,160)):
    with Image.open(path) as im:
        rgb=ImageOps.exif_transpose(im).convert('RGB').resize(size,Image.Resampling.LANCZOS)
        return np.asarray(rgb,dtype=np.uint8)

def compare_visually(path_a,path_b):
    pixel_same=decoded_pixel_hash(path_a)==decoded_pixel_hash(path_b)
    a=normalized_pixels(path_a); b=normalized_pixels(path_b)
    mae=float(np.abs(a.astype(np.int16)-b.astype(np.int16)).mean()/255.)
    score=float(structural_similarity(a,b,channel_axis=2,data_range=255)) if SKIMAGE_AVAILABLE else None
    return pixel_same,score,mae

VISUAL_EXAMINATION_ERRORS=[]
for pair in CROSS_PAIR_ROWS:
    try:
        path_a=D3_ROW_BY_REL[pair['d3_relative_path']]['external_absolute_path']
        path_b=D1_ROW_BY_REL[pair['d1_reference_path']]['reference_absolute_path']
        same,ssim,mae=compare_visually(path_a,path_b)
        pair.update(pixel_identity=same,visual_comparison_ssim=ssim,visual_comparison_rgb_mae=mae)
        if same:
            D3_REASONS[pair['d3_relative_path']].add('EXCLUDE_CONFIRMED_DECODED_PIXEL_D1_OVERLAP')
    except Exception as exc:
        VISUAL_EXAMINATION_ERRORS.append(pair['pair_id']+': '+repr(exc))
for pair in INTERNAL_PAIR_ROWS:
    try:
        path_a=D3_ROW_BY_REL[pair['d3_image_a']]['external_absolute_path']
        path_b=D3_ROW_BY_REL[pair['d3_image_b']]['external_absolute_path']
        same,ssim,mae=compare_visually(path_a,path_b)
        pair.update(pixel_identity=same,visual_comparison_ssim=ssim,visual_comparison_rgb_mae=mae)
        if same:
            # Preserve the first lexical record, withhold the second.
            redundant=max(pair['d3_image_a'],pair['d3_image_b'])
            D3_REASONS[redundant].add('EXCLUDE_CONFIRMED_DECODED_PIXEL_INTERNAL_D3_COPY')
    except Exception as exc:
        VISUAL_EXAMINATION_ERRORS.append(pair['pair_id']+': '+repr(exc))

def make_thumbnail(path,size=(210,155)):
    try:
        with Image.open(path) as im:
            return ImageOps.pad(ImageOps.exif_transpose(im).convert('RGB'),size,method=Image.Resampling.LANCZOS,color='white')
    except Exception:
        return Image.new('RGB',size,'#ffe5e5')

def sheet_group(pairs,mode):
    folder=VISUAL_DIR/mode
    folder.mkdir(parents=True,exist_ok=True)
    page_count=math.ceil(len(pairs)/PAIRS_PER_SHEET)
    if page_count>MAX_CONTACT_SHEETS:
        return 0,False
    for page_idx in range(page_count):
        batch=pairs[page_idx*PAIRS_PER_SHEET:(page_idx+1)*PAIRS_PER_SHEET]
        canvas=Image.new('RGB',(930,40+len(batch)*200),'white')
        draw=ImageDraw.Draw(canvas)
        draw.text((15,12),f'{mode} suspected pairs · sheet {page_idx+1}/{page_count}',fill='black')
        for row_idx,pair in enumerate(batch):
            if mode=='cross_d1':
                left=D3_ROW_BY_REL[pair['d3_relative_path']]['external_absolute_path']
                right=D1_ROW_BY_REL[pair['d1_reference_path']]['reference_absolute_path']
                ltag='D3 '+pair['d3_relative_path']
                rtag='D1 '+pair['d1_reference_path']
            else:
                left=D3_ROW_BY_REL[pair['d3_image_a']]['external_absolute_path']
                right=D3_ROW_BY_REL[pair['d3_image_b']]['external_absolute_path']
                ltag='D3 '+pair['d3_image_a']
                rtag='D3 '+pair['d3_image_b']
            y=42+row_idx*200
            canvas.paste(make_thumbnail(left),(12,y+24))
            canvas.paste(make_thumbnail(right),(470,y+24))
            draw.text((12,y+2),f"{pair['pair_id']} · pHash Hamming={pair['phash_hamming']} · identical pixels={pair['pixel_identity']}",fill='black')
            draw.text((12,y+184),ltag[-68:],fill='black')
            draw.text((470,y+184),rtag[-68:],fill='black')
        canvas.save(folder/f'{mode}_{page_idx+1:04d}.jpg',quality=86)
    return page_count,True

CROSS_SHEETS,CROSS_SHEETS_FULL=sheet_group(CROSS_PAIR_ROWS,'cross_d1')
INTERNAL_SHEETS,INTERNAL_SHEETS_FULL=sheet_group(INTERNAL_PAIR_ROWS,'within_d3')
print('Machine-assisted comparisons complete:',not VISUAL_EXAMINATION_ERRORS)
print('Contact-sheet coverage complete:',CROSS_SHEETS_FULL and INTERNAL_SHEETS_FULL)
print('Generated contact sheets:',CROSS_SHEETS+INTERNAL_SHEETS)
print('scikit-image SSIM available:',SKIMAGE_AVAILABLE)
if VISUAL_EXAMINATION_ERRORS:print('First 5 examination failures:',VISUAL_EXAMINATION_ERRORS[:5])


Machine-assisted comparisons complete: True
Contact-sheet coverage complete: True
Generated contact sheets: 10
scikit-image SSIM available: True


## 8. Review decisions, exclusions and class mapping

We **do not infer human confirmation** of near-duplicate image pairs from the earlier D3 class-definition confirmation. This notebook excludes exact overlaps, marks all near-image alerts as withheld, and creates a review queue. Images not flagged are eligible. A conservative screened manifest may be locked if all D1 references are available and the complete screening succeeds.

In [9]:
# 8 — Assemble per-image decisions without dropping exclusion evidence
CONFIRMED_EXCLUSION_PREFIXES=('EXCLUDE_',)

D3_FINAL_ROWS=[]
for r in D3_SELECTED:
    rel=r['external_relative_path']
    reasons=sorted(D3_REASONS.get(rel,set()))
    hard_exclude=any(v.startswith(CONFIRMED_EXCLUSION_PREFIXES) for v in reasons)
    near_withhold=any(v.startswith('WITHHOLD_') for v in reasons)
    if hard_exclude:
        status='EXCLUDED_CONFIRMED_OR_INVALID'
    elif near_withhold:
        status='WITHHELD_NEAR_DUPLICATE_CANDIDATE'
    else:
        status='ELIGIBLE_AFTER_REFERENCE_VERIFICATION'
    D3_FINAL_ROWS.append({
        'dataset':'D3_RealWaste','external_relative_path':rel,
        'external_absolute_path':r['external_absolute_path'],
        'filename':r['filename'], 'source_category':r['source_category'],
        'class':r['class'],'output_index':r['output_index'],
        'sha256':r['sha256'], 'phash_hex':r['phash_hex'],
        'image_ok':r['image_ok'],'image_width':r['width'],'image_height':r['height'],
        'decode_error':r['decode_error'], 'final_status':status,
        'exclusion_reason':';'.join(reasons),
        'human_review_of_near_pairs':'NOT_HUMAN_REVIEWED' if near_withhold else 'NOT_REQUIRED',
        'class_mapping_human_review':D3_CLASS_MAPPING_HUMAN_REVIEW
    })

CLASS_MAPPING_ROWS=[]
for source in ALL_D3_CLASSES:
    model=D3_SOURCE_TO_MODEL.get(source,'')
    CLASS_MAPPING_ROWS.append({
        'd3_source_label':source,
        'remnet_trained_class':model,
        'remnet_output_index':CLASS_TO_INDEX[model] if model else '',
        'include_for_primary_d3':bool(model),
        'observed_d3_files':D3_COUNTS[source],
        'class_review':'CONFIRMED_BY_USER' if model else 'NOT_APPLICABLE',
        'class_mapping_basis':'Six-output trained class index order; Miscellaneous Trash maps to Trash' if model else 'No corresponding output'
    })

print('Selected six-class images:',len(D3_FINAL_ROWS))
print('Disposition counts:',dict(Counter(r['final_status'] for r in D3_FINAL_ROWS)))
print('Per-class eligible counts:',dict(Counter(r['class'] for r in D3_FINAL_ROWS
                                              if r['final_status']=='ELIGIBLE_AFTER_REFERENCE_VERIFICATION')))
print('User-confirmed class mapping is distinct from unverified near-duplicate human review.')

Selected six-class images: 3587
Disposition counts: {'ELIGIBLE_AFTER_REFERENCE_VERIFICATION': 3497, 'WITHHELD_NEAR_DUPLICATE_CANDIDATE': 90}
Per-class eligible counts: {'cardboard': 454, 'glass': 419, 'metal': 757, 'paper': 500, 'plastic': 889, 'trash': 478}
User-confirmed class mapping is distinct from unverified near-duplicate human review.


## 9. Gate and fingerprint the D3 evaluation manifest

Do not load any old manifest. After screening, create the **new D3 output manifest** from the filtered RealWaste image rows only if both source-image inventories are complete, no flagged image survives, and required class mapping is present. This locks the evaluation subset for every model and seed. Original D1 split membership remains `UNKNOWN` unless directly encoded in folder names.


In [10]:
# 9 — Defensible fail-closed lock conditions
BLOCKERS=[]
if not D1_REFERENCE_READY:
    BLOCKERS.append('Original D1 source images or their image-integrity checks are incomplete')
if not PHASH_CROSS_COMPLETE:
    BLOCKERS.append('Cross-D1 pHash examination incomplete')
if not PHASH_INTERNAL_COMPLETE:
    BLOCKERS.append('Within-D3 pHash examination incomplete')
if VISUAL_EXAMINATION_ERRORS:
    BLOCKERS.append(f'{len(VISUAL_EXAMINATION_ERRORS)} image-pair visual computations failed')
if not (CROSS_SHEETS_FULL and INTERNAL_SHEETS_FULL):
    BLOCKERS.append('Contact-sheet output would be incomplete; raise MAX_CONTACT_SHEETS and rerun')
if len(D3_VALID)!=len(D3_SELECTED)-len(D3_BAD):
    BLOCKERS.append('D3 integrity accounting mismatch')
if sum(D3_COUNTS.values())!=len(D3_ROWS):
    BLOCKERS.append('Nine-class D3 inventory accounting mismatch')
if set(D3_SOURCE_TO_MODEL.values())!=set(MODEL_CLASSES) or len(CLASS_TO_INDEX)!=6:
    BLOCKERS.append('Trained six-class mapping inconsistent')
if not D3_CLASS_ROOT.exists():
    BLOCKERS.append('RealWaste input missing')
if not all(D3_COUNTS[x]>0 for x in D3_SOURCE_TO_MODEL):
    BLOCKERS.append('Missing one or more required D3 categories')

ELIGIBLE=[r for r in D3_FINAL_ROWS if r['final_status']=='ELIGIBLE_AFTER_REFERENCE_VERIFICATION']
if not all(any(x['class']==c for x in ELIGIBLE) for c in MODEL_CLASSES):
    BLOCKERS.append('One or more classes absent from conservatively screened D3 candidates')
if len({r['sha256'] for r in ELIGIBLE})!=len(ELIGIBLE):
    BLOCKERS.append('Eligible D3 contains duplicate raw SHA-256')
if any(r['sha256'] in D1_BY_SHA for r in ELIGIBLE):
    BLOCKERS.append('Unexcluded SHA-256 overlap with original D1')
if any(r['exclusion_reason'] for r in ELIGIBLE):
    BLOCKERS.append('Eligibility and exclusions inconsistent')

FINAL_LOCKED=(not BLOCKERS)
LOCKED_FILE=OUT/'D3_Section2_Final_Automated_Screened_Six_Class_Manifest.csv'
if LOCKED_FILE.exists():
    LOCKED_FILE.unlink()  # prevent stale locked manifests from previous executions

# Stable fingerprint includes row order, relative file path, trained index, source SHA-256.
LOCKED_FINGERPRINT=hashlib.sha256(''.join(
    f"{r['external_relative_path']}|{r['output_index']}|{r['sha256']}\n"
    for r in sorted(ELIGIBLE,key=lambda x:x['external_relative_path'])
).encode('utf-8')).hexdigest() if FINAL_LOCKED else None

print('LOCKED AUTOMATED-SCREENED SIX-CLASS D3:',FINAL_LOCKED)
print('Final count (if locked):',len(ELIGIBLE) if FINAL_LOCKED else 'NOT LOCKED')
print('Selected-manifest fingerprint:',LOCKED_FINGERPRINT or 'NOT AVAILABLE')
for blocker in BLOCKERS:print('BLOCKER:',blocker)


LOCKED AUTOMATED-SCREENED SIX-CLASS D3: True
Final count (if locked): 3497
Selected-manifest fingerprint: 0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0


## 10. Export all audit tables, provenance, review evidence and one ZIP

The full inventory records exclusions and class labels; the final CSV is written only when verified. Source metadata below records the official UCI RealWaste DOI/license and the attached Kaggle mirror location. The **exact Kaggle mirror version remains unverified unless separately documented**, so the content-derived fingerprint is provided for reproducibility.

In [11]:
# 10 — Persist full-precision records, reproducibility metadata, and output ZIP

def write_csv(filename,rows,columns=None):
    path=OUT/filename
    if rows:
        pd.DataFrame(rows).to_csv(path,index=False,float_format='%.12f',encoding='utf-8')
    elif columns:
        pd.DataFrame(columns=columns).to_csv(path,index=False,encoding='utf-8')
    else:
        pd.DataFrame().to_csv(path,index=False,encoding='utf-8')
    return path

write_csv('D3_Section2_Full_Nine_Class_Inventory.csv',D3_ROWS)
write_csv('D3_Section2_Original_D1_Reference_Inventory.csv',D1_ROWS)
write_csv('D3_Section2_Exact_SHA256_Overlaps_With_Full_D1_Reference.csv',EXACT_D1_REFERENCE_OVERLAPS)
write_csv('D3_Section2_Exact_Internal_D3_Duplicate_Groups.csv',
          [{'sha256':k,'d3_paths':';'.join(x['external_relative_path'] for x in rs),'n':len(rs)}
           for k,rs in D3_EXACT_INTERNAL.items()])
write_csv('D3_Section2_Near_Duplicate_D3_vs_D1_Candidates.csv',CROSS_PAIR_ROWS)
write_csv('D3_Section2_Near_Duplicate_Within_D3_Candidates.csv',INTERNAL_PAIR_ROWS)
write_csv('D3_Section2_Complete_Six_Class_Manifest_With_Exclusions.csv',D3_FINAL_ROWS)
write_csv('D3_Section2_Exclusions_And_Withheld.csv',[x for x in D3_FINAL_ROWS if x['final_status']!='ELIGIBLE_AFTER_REFERENCE_VERIFICATION'])
write_csv('D3_Section2_Provisional_Eligible_Six_Class_Manifest.csv',ELIGIBLE)
write_csv('D3_Section2_Class_Mapping.csv',CLASS_MAPPING_ROWS)
write_csv('D3_Section2_Class_Distribution.csv',[
    {'category':c,'observed_files':D3_COUNTS[c],
     'published_reference_files':PUBLISHED_D3_COUNTS[c],
     'selected_for_six_class_evaluation':c in D3_SOURCE_TO_MODEL,
     'eligible_after_screening':sum(x['source_category']==c for x in ELIGIBLE)}
    for c in ALL_D3_CLASSES
])
write_csv('D3_Section2_Human_Review_Queue.csv',[
    {'pair_id':r['pair_id'],'pair_type':'D3_vs_D1','d3_image_a':r['d3_relative_path'],
     'other_image':r['d1_reference_path'],'phash_hamming':r['phash_hamming'],
     'machine_identical_pixels':r['pixel_identity'],'human_review_decision':'PENDING',
     'human_review_notes':''} for r in CROSS_PAIR_ROWS
]+[
    {'pair_id':r['pair_id'],'pair_type':'Within_D3','d3_image_a':r['d3_image_a'],
     'other_image':r['d3_image_b'],'phash_hamming':r['phash_hamming'],
     'machine_identical_pixels':r['pixel_identity'],'human_review_decision':'PENDING',
     'human_review_notes':''} for r in INTERNAL_PAIR_ROWS
])

if FINAL_LOCKED:
    write_csv(LOCKED_FILE.name, sorted(ELIGIBLE,key=lambda x:x['external_relative_path']))

PROVENANCE={
    'evaluation_dataset':'D3_RealWaste',
    'official_dataset_name':'RealWaste',
    'official_source_url':'https://archive.ics.uci.edu/dataset/908/realwaste',
    'official_dataset_DOI':'10.24432/C5SS4G',
    'citation':'Single, S., Iranmanesh, S., & Raad, R. (2023). RealWaste [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5SS4G',
    'official_UCI_license':'CC BY 4.0',
    'mounted_Kaggle_mirror':'https://www.kaggle.com/datasets/akash264208/real-waste',
    'actual_D3_mount_path':str(D3_INPUT_PATH),
    'detected_D3_class_root':str(D3_CLASS_ROOT),
    'mirror_version':'NOT VERIFIED FROM ATTACHED MOUNT',
    'download_method':'Mounted Kaggle dataset input; no network downloads in this notebook',
    'dataset_snapshot_SHA256':D3_INVENTORY_FINGERPRINT,
    'published_9_class_count':sum(PUBLISHED_D3_COUNTS.values()),
    'observed_9_class_count':sum(D3_COUNTS.values()),
    'published_6_class_candidate_count':sum(PUBLISHED_D3_COUNTS[x] for x in D3_SOURCE_TO_MODEL),
    'observed_6_class_candidate_count':len(D3_SELECTED),
    'observed_6_class_readable_count':len(D3_VALID),
    'observed_counts_match_official_release':COUNT_MATCHES_PUBLIC,
    'user_confirmed_class_mapping':D3_CLASS_MAPPING_HUMAN_REVIEW,
    'external_trash_source_category':'Miscellaneous Trash',
    'original_D1_reference_root':str(D1_INPUT_PATH),
    'original_D1_reference_basis':'ALL IMAGE FILES DIRECTLY ENUMERATED FROM ATTACHED D1 SOURCE; no split manifest required',
    'original_D1_expected_historical_splits_for_information_only':EXPECTED_D1_SPLITS,
    'original_D1_split_labels_inferred_from_folders':dict(D1_SPLIT_OBSERVED),
    'original_D1_historical_split_membership_verified_from_folders':D1_SPLIT_PARTITION_VERIFIED,
    'original_D1_split_assignment_limitation':'No split CSV supplied; unmarked paths have UNKNOWN split and are not assigned artificially',
    'original_D1_reference_ready':D1_REFERENCE_READY,
    'd3_eligible_count':len(ELIGIBLE),
    'd3_exact_original_source_overlap_pairs':len(EXACT_D1_REFERENCE_OVERLAPS),
    'd3_internal_exact_duplicate_groups':len(D3_EXACT_INTERNAL),
    'd3_vs_original_phash_candidate_pairs':len(CROSS_PAIR_ROWS),
    'd3_internal_phash_candidate_pairs':len(INTERNAL_PAIR_ROWS),
    'phash_threshold':PHASH_HAMMING_THRESHOLD,
    'phash_implementation':'PIL grayscale 32x32; scipy.fft.dctn type2 norm ortho; 8x8 block; median excluding DC',
    'machine_visual_evidence_available':not VISUAL_EXAMINATION_ERRORS,
    'near_duplicate_human_review_status':'NOT CLAIMED; contact sheets and review queue provided',
    'conservative_unadjudicated_phash_policy':'Withhold all flagged candidate images; exact/pixel-confirmed overlaps excluded',
    'final_automated_screened_d3_manifest_locked':FINAL_LOCKED,
    'final_manifest_name':LOCKED_FILE.name if FINAL_LOCKED else None,
    'final_manifest_SHA256':LOCKED_FINGERPRINT,
    'blockers':BLOCKERS,
    'environment':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'PIL':Image.__version__},
    'audit_UTC':datetime.datetime.now(datetime.timezone.utc).isoformat()
}
(OUT/'D3_Section2_Source_License_Citation_Version_Download_Details.json').write_text(json.dumps(PROVENANCE,indent=2),encoding='utf-8')
(OUT/'D3_Section2_Verification_Summary.json').write_text(json.dumps(PROVENANCE,indent=2),encoding='utf-8')

report=[
    'REMNET REALWASTE D3 — SECTION 2 INDEPENDENCE AUDIT',
    'D3 external: '+str(D3_INPUT_PATH),
    'D1 ORIGINAL reference: '+str(D1_INPUT_PATH),
    'Input manifest: NONE (D1 images read directly)',
    'D1 folder-inferred split counts: '+str(dict(D1_SPLIT_OBSERVED)),
    'D3 9-class total: '+str(sum(D3_COUNTS.values())),
    'D3 selected 6-class total: '+str(len(D3_SELECTED)),
    'D3 valid: '+str(len(D3_VALID)),
    'Exact cross-D1 source pairs: '+str(len(EXACT_D1_REFERENCE_OVERLAPS)),
    'Cross-D1 pHash candidate pairs: '+str(len(CROSS_PAIR_ROWS)),
    'Within-D3 pHash candidate pairs: '+str(len(INTERNAL_PAIR_ROWS)),
    'Automatically screened candidates eligible: '+str(len(ELIGIBLE)),
    'LOCKED: '+str(FINAL_LOCKED),
    'Locked manifest fingerprint: '+str(LOCKED_FINGERPRINT),
    'BLOCKERS: '+('; '.join(BLOCKERS) if BLOCKERS else 'NONE'),
    'No D2 dataset used.',
    'All models/seeds must reuse this exact D3 locked manifest. Not a claim of human near-pair adjudication.'
]
(OUT/'D3_Section2_Verification_Report.txt').write_text('\n'.join(report)+'\n',encoding='utf-8')

ZIP_PATH=OUT.parent/'REMNET_D3_Section2_RealWaste_Independence_All_Outputs.zip'
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for file in sorted(OUT.rglob('*')):
        if file.is_file():
            z.write(file,file.relative_to(OUT.parent).as_posix())
print('\n'.join(report))
print('Results ZIP:',ZIP_PATH)
print('Results ZIP bytes:',ZIP_PATH.stat().st_size)


REMNET REALWASTE D3 — SECTION 2 INDEPENDENCE AUDIT
D3 external: /kaggle/input/datasets/akash264208/real-waste
D1 ORIGINAL reference: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset
Input manifest: NONE (D1 images read directly)
D1 folder-inferred split counts: {'Testing': 464, 'Training': 2508}
D3 9-class total: 4752
D3 selected 6-class total: 3587
D3 valid: 3587
Exact cross-D1 source pairs: 0
Cross-D1 pHash candidate pairs: 10
Within-D3 pHash candidate pairs: 58
Automatically screened candidates eligible: 3497
LOCKED: True
Locked manifest fingerprint: 0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0
BLOCKERS: NONE
No D2 dataset used.
All models/seeds must reuse this exact D3 locked manifest. Not a claim of human near-pair adjudication.
Results ZIP: /kaggle/working/REMNET_D3_Section2_RealWaste_Independence_All_Outputs.zip
Results ZIP bytes: 2273265


## How to proceed

1. In Kaggle attach **only** RealWaste D3 at `/kaggle/input/datasets/akash264208/real-waste` and original D1 at `/kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset`.
2. Run all cells from the beginning; no input manifest file is required.
3. Download `REMNET_D3_Section2_RealWaste_Independence_All_Outputs.zip` from Kaggle outputs.
4. Inspect the summary, contact sheets, source-image overlap tables, D3 exclusions, class mapping and **newly generated** `D3_Section2_Final_Automated_Screened_Six_Class_Manifest.csv`.
5. If D1 source folders do not identify Train/Validation/Test, the notebook reports split membership as `UNKNOWN`, while still checking overlaps against **all original source images**. Do not claim partition-specific duplicate counts without a split record.
6. Use the newly generated locked D3 manifest (not the original source tree) for **all** ResNet50, MobileNetV2, EfficientNet-Lite0 and REMNET external evaluations.

**The workflow preserves the automatic SHA-256, pHash, visual evidence, conservative exclusion, fixed class mapping, provenance and results ZIP structure from the supplied Section-2 reference.**
